<a href="https://colab.research.google.com/github/sadafmaqbool074-bot/production-planning-optimization-pyomo/blob/main/notebooks/production_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Production Planning Problem

A manufacturing company produces 10 different products (P1–P10). The company must decide how many units of each product to manufacture during the next planning period.

Each product has a different:

Profit per unit
Machine-hour requirement per unit
Labour-hour requirement per unit
Raw-material requirement per unit
Maximum market demand

The factory has limited resources:

3,000 machine hours
2,500 labour hours
4,000 kg of raw material

The company cannot produce more than the maximum market demand for any product.

Question

How many units of each of the 10 products should the company produce to maximise total profit while satisfying all machine, labour, raw-material, and market-demand constraints?

In [ ]:
import pandas as pd

product = pd.read_excel("Products.csv")
print(product)

  product  profit_per_unit  machine_hours_per_unit  labour_hours_per_unit  \
0      P1               40                     2.0                    1.5   
1      P2               35                     1.5                    2.0   
2      P3               55                     3.0                    2.5   
3      P4               30                     1.0                    1.0   
4      P5               70                     4.0                    3.0   
5      P6               45                     2.5                    2.0   
6      P7               60                     3.0                    1.5   
7      P8               50                     2.0                    3.0   
8      P9               80                     4.5                    3.5   
9     P10               38                     1.5                    1.0   

   material_kg_per_unit  max_demand  
0                     3         500  
1                     2         400  
2                     4         300  


In [ ]:
!pip install pyomo
!apt-get install -y -qq glpk-utils
import pandas as pd
import pyomo.environ as pyo

product = pd.read_excel("Products.csv")

print(product)

model = pyo.ConcreteModel()

model.PRODUCTS = pyo.Set(
    initialize=product["product"].tolist()
)

print(list(model.PRODUCTS))


  product  profit_per_unit  machine_hours_per_unit  labour_hours_per_unit  \
0      P1               40                     2.0                    1.5   
1      P2               35                     1.5                    2.0   
2      P3               55                     3.0                    2.5   
3      P4               30                     1.0                    1.0   
4      P5               70                     4.0                    3.0   
5      P6               45                     2.5                    2.0   
6      P7               60                     3.0                    1.5   
7      P8               50                     2.0                    3.0   
8      P9               80                     4.5                    3.5   
9     P10               38                     1.5                    1.0   

   material_kg_per_unit  max_demand  
0                     3         500  
1                     2         400  
2                     4         300  


In [ ]:
# Parameter
model.profit=pyo.Param(model.PRODUCTS,initialize=product.set_index("product")["profit_per_unit"].to_dict())
model.machine_hours=pyo.Param(model.PRODUCTS,initialize=product.set_index("product")["machine_hours_per_unit"].to_dict())
model.labour_hours=pyo.Param(model.PRODUCTS,initialize=product.set_index("product")["labour_hours_per_unit"].to_dict())
model.material=pyo.Param(model.PRODUCTS,initialize=product.set_index("product")["material_kg_per_unit"].to_dict())
model.max_demand=pyo.Param(model.PRODUCTS,initialize=product.set_index("product")["max_demand"].to_dict())

In [ ]:
# Decision Variable
model.x=pyo.Var(model.PRODUCTS,within=pyo.NonNegativeReals)
#Objective Function
def Objective_rule(model):
  return sum(model.profit[p]*model.x[p] for p in model.PRODUCTS)
model.Objective = pyo.Objective(rule=Objective_rule,sense=pyo.maximize)

#Constraint
def machine_constraint_rule(model):
    return sum(
        model.machine_hours[p] * model.x[p]
        for p in model.PRODUCTS
    ) <= 3000
model.machine_constraint = pyo.Constraint(
    rule=machine_constraint_rule
)
def labour_constraint_rule(model):
    return sum(
        model.labour_hours[p] * model.x[p]
        for p in model.PRODUCTS
    ) <= 2500

model.labour_constraint = pyo.Constraint(
    rule=labour_constraint_rule
)
def material_constraint_rule(model):
    return sum(
        model.material[p] * model.x[p]
        for p in model.PRODUCTS
    ) <= 4000

model.material_constraint = pyo.Constraint(
    rule=material_constraint_rule
)
def demand_constraint_rule(model, p):
    return model.x[p] <= model.max_demand[p]

model.demand_constraint = pyo.Constraint(
    model.PRODUCTS,
    rule=demand_constraint_rule
)


#Solver
solver = pyo.SolverFactory(
    "glpk",
    executable="/usr/bin/glpsol"
)
print(solver.available())

results = solver.solve(model, tee=True)

print(results.solver.status)
print(results.solver.termination_condition)
print(results.solver.status)
print(results.solver.termination_condition)


This is usually indicative of a modelling error.
To avoid this warning, use block.del_component() and block.add_component().
This is usually indicative of a modelling error.
To avoid this warning, use block.del_component() and block.add_component().
This is usually indicative of a modelling error.
To avoid this warning, use block.del_component() and block.add_component().
This is usually indicative of a modelling error.
To avoid this warning, use block.del_component() and block.add_component().
This is usually indicative of a modelling error.
To avoid this warning, use block.del_component() and block.add_component().
This is usually indicative of a modelling error.
To avoid this warning, use block.del_component() and block.add_component().


True
GLPSOL--GLPK LP/MIP Solver 5.0
Parameter(s) specified in the command line:
 --write /tmp/tmpnsg5xl4e.glpk.raw --wglp /tmp/tmpbz0j02l0.glpk.glp --cpxlp
 /tmp/tmp757mzzpm.pyomo.lp
Reading problem data from '/tmp/tmp757mzzpm.pyomo.lp'...
13 rows, 10 columns, 40 non-zeros
108 lines were read
Writing problem data to '/tmp/tmpbz0j02l0.glpk.glp'...
89 lines were written
GLPK Simplex Optimizer 5.0
13 rows, 10 columns, 40 non-zeros
Preprocessing...
3 rows, 10 columns, 30 non-zeros
Scaling...
 A: min|aij| =  1.000e+00  max|aij| =  6.000e+00  ratio =  6.000e+00
Problem data seem to be well scaled
Constructing initial basis...
Size of triangular part is 3
*     0: obj =  -0.000000000e+00 inf =   0.000e+00 (10)
*    12: obj =   6.683333333e+04 inf =   0.000e+00 (0)
OPTIMAL LP SOLUTION FOUND
Time used:   0.0 secs
Memory used: 0.0 Mb (40513 bytes)
Writing basic solution to '/tmp/tmpnsg5xl4e.glpk.raw'...
32 lines were written
ok
optimal
ok
optimal


In [ ]:
import pandas as pd
results_table = pd.DataFrame({
    "Product": list(model.PRODUCTS),
    "Production": [pyo.value(model.x[p]) for p in model.PRODUCTS],
    "Profit per Unit": [pyo.value(model.profit[p]) for p in model.PRODUCTS],
    "Total Profit": [
        pyo.value(model.profit[p] * model.x[p])
        for p in model.PRODUCTS
    ]
})

results_table

,Product,Production,Profit per Unit,Total Profit
0,P1,0.000000,40,0.000000
1,P2,400.000000,35,14000.000000
2,P3,0.000000,55,0.000000
3,P4,250.000000,30,7500.000000
4,P5,0.000000,70,0.000000
5,P6,0.000000,45,0.000000
6,P7,300.000000,60,18000.000000
7,P8,166.666667,50,8333.333333
8,P9,0.000000,80,0.000000
9,P10,500.000000,38,19000.000000
